# Baby Name Visualisations in France (1900–2020)

This notebook implements the **3 interactive visualisations** for the Week 2 project, based on the INSEE baby names dataset by department.

| # | Theme | Visualisation type |
|---|---|---|
| 1 | Temporal evolution | Interactive line chart (click on legend) |
| 2 | Regional effect | Choropleth map with name selector |
| 3 | Gender effects | M/F proportion chart with name selector |

**Environment**: Poetry + kernel `visualisation-prenoms` (Python 3.12)

## Poetry Environment

The project uses Poetry for dependency management. The Jupyter kernel `visualisation-prenoms` was registered via:

```bash
# From the project directory
poetry init --no-interaction --name "visualisation-prenoms" --python "^3.12"
poetry add altair pandas geopandas jupyter ipykernel vega_datasets
poetry run python -m ipykernel install --user --name="visualisation-prenoms" --display-name="Python (visualisation-prenoms)"
```

Select the **Python (visualisation-prenoms)** kernel in VS Code before running the cells.

In [1]:
import altair as alt
import pandas as pd
import geopandas as gpd

# Allow Altair to handle large datasets (>5000 rows)
alt.data_transformers.enable('json')

print(f"altair   : {alt.__version__}")
print(f"pandas   : {pd.__version__}")
print(f"geopandas: {gpd.__version__}")

altair   : 6.1.0
pandas   : 3.0.3
geopandas: 1.1.3


## Data Loading and Cleaning

The `dpt2020.csv` file contains ~3.7 M rows. We remove:
- Aggregated rare names (`_PRENOMS_RARES`)
- Unknown departments (`XX`)
- Unknown years (`XXXX`)

In [2]:
# --- Load baby names ---
names = pd.read_csv("dpt2020.csv", sep=";", dtype=str)

# Cleaning
names = names[names['preusuel'] != '_PRENOMS_RARES']
names = names[names['dpt'] != 'XX']
names = names[names['annais'] != 'XXXX']

names['annais'] = names['annais'].astype(int)
names['nombre'] = pd.to_numeric(names['nombre'], errors='coerce')
names['sexe']   = names['sexe'].astype(int)
names.dropna(subset=['nombre'], inplace=True)
names['nombre'] = names['nombre'].astype(int)

# --- Load geographic boundaries ---
depts = gpd.read_file('departements-version-simplifiee.geojson')

print(f"Names loaded : {len(names):,} rows")
print(f"Departments  : {len(depts)}")
names.sample(3)

Names loaded : 3,668,274 rows
Departments  : 96


,sexe,preusuel,annais,dpt,nombre
719990,1,HUGUES,1965,971,28
2350118,2,ELSA,1996,972,4
765976,1,JANICK,1953,60,6


---
## Visualisation 1 — Temporal Evolution of Baby Names

**Questions:** How do baby names evolve over time? Are some names consistently popular? Did any names experience a sudden, brief peak in popularity?

**Design choice:** Interactive line chart showing the 20 most popular names.  
→ Click a name in the legend to highlight it (others fade out).

In [3]:
# National aggregation by (name, year)
names_time = names.groupby(['preusuel', 'annais'], as_index=False)['nombre'].sum()

# Top 20 names across all years
top20 = (
    names_time.groupby('preusuel')['nombre']
    .sum()
    .nlargest(20)
    .index.tolist()
)
viz1_data = names_time[names_time['preusuel'].isin(top20)].copy()

print(f"Viz 1 data: {len(viz1_data)} rows ({len(top20)} names × years)")

Viz 1 data: 2362 rows (20 names × years)


In [4]:
# Interactive selection via legend click
legend_selection = alt.selection_point(fields=['preusuel'], bind='legend')

# X-axis: one tick every 10 years
x_ticks = list(range(1900, 2021, 10))

chart_viz1 = (
    alt.Chart(viz1_data)
    .mark_line(point=alt.OverlayMarkDef(size=30))
    .encode(
        x=alt.X(
            'annais:O',
            title='Year',
            axis=alt.Axis(values=x_ticks, labelAngle=-45),
        ),
        y=alt.Y('nombre:Q', title='Number of births'),
        color=alt.Color(
            'preusuel:N',
            title='Name',
            legend=alt.Legend(title='Click to filter'),
        ),
        opacity=alt.condition(legend_selection, alt.value(1.0), alt.value(0.06)),
        strokeWidth=alt.condition(legend_selection, alt.value(2.5), alt.value(0.8)),
        tooltip=[
            alt.Tooltip('preusuel:N', title='Name'),
            alt.Tooltip('annais:O',   title='Year'),
            alt.Tooltip('nombre:Q',   title='Births', format=','),
        ],
    )
    .add_params(legend_selection)
    .properties(
        width=860,
        height=420,
        title=alt.TitleParams(
            'Temporal evolution of the 20 most popular baby names in France (1900–2020)',
            fontSize=14,
        ),
    )
)

chart_viz1

alt.Chart(...)

---
## Visualisation 2 — Regional Effect (Choropleth Map)

**Questions:** Are some names more popular in certain regions? Are popular names uniformly popular across the whole country?

**Design choice:** Choropleth map with a dropdown menu to select the name.  
→ The colour encodes births **per 1,000 births** in the department (normalised to correct for population size).

In [5]:
# Aggregation by (department, name) across all years
grouped_geo = names.groupby(['dpt', 'preusuel'], as_index=False)['nombre'].sum()

# Total births per department for normalisation
dept_totals = (
    names.groupby('dpt', as_index=False)['nombre']
    .sum()
    .rename(columns={'nombre': 'total_dept'})
)
grouped_geo = grouped_geo.merge(dept_totals, on='dpt')
grouped_geo['pour_mille'] = (grouped_geo['nombre'] / grouped_geo['total_dept'] * 1000).round(3)

# Top 60 names for the dropdown menu
top60 = (
    grouped_geo.groupby('preusuel')['nombre']
    .sum()
    .nlargest(60)
    .index.sort_values()
    .tolist()
)

# Merge with geographic boundaries (keep geopandas dataframe for geometry)
viz2_data = depts.merge(
    grouped_geo[grouped_geo['preusuel'].isin(top60)],
    how='left',
    left_on='code',
    right_on='dpt',
)

print(f"Viz 2 data: {len(viz2_data)} rows | {len(top60)} names in the dropdown")

Viz 2 data: 5642 rows | 60 names in the dropdown


In [6]:
# Dropdown to select the name
dropdown2 = alt.binding_select(options=top60, name='Name: ')
name_select2 = alt.selection_point(
    fields=['preusuel'],
    bind=dropdown2,
    value=top60[0],
)

chart_viz2 = (
    alt.Chart(viz2_data)
    .mark_geoshape(stroke='white', strokeWidth=0.5)
    .encode(
        color=alt.Color(
            'pour_mille:Q',
            scale=alt.Scale(scheme='blues'),
            title='Births per 1,000',
            legend=alt.Legend(gradientLength=220),
        ),
        tooltip=[
            alt.Tooltip('nom:N',        title='Department'),
            alt.Tooltip('preusuel:N',   title='Name'),
            alt.Tooltip('nombre:Q',     title='Total births', format=','),
            alt.Tooltip('pour_mille:Q', title='Per 1,000 births', format='.2f'),
        ],
    )
    .add_params(name_select2)
    .transform_filter(name_select2)
    .project(type='mercator')
    .properties(
        width=680,
        height=580,
        title=alt.TitleParams(
            'Regional distribution of a name (births per 1,000 in the department)',
            fontSize=14,
        ),
    )
)

chart_viz2

alt.Chart(...)

---
## Visualisation 3 — Gender Effects (Proportion Chart)

**Questions:** Are there gender effects in the data? Does the popularity of names given to both sexes evolve consistently?

**Design choice:** Stacked 100% area chart showing the M/F breakdown of a name over time.  
→ Dropdown to choose from **unisex names** (given to both sexes at more than 5%).  
→ The dashed line at 50% serves as a visual reference to spot gender switches.

In [7]:
# National aggregation by (name, year, sex)
gender_time = names.groupby(['preusuel', 'annais', 'sexe'], as_index=False)['nombre'].sum()

# Pivot: one column per sex
pivot = (
    gender_time
    .pivot_table(index=['preusuel', 'annais'], columns='sexe', values='nombre', fill_value=0)
    .reset_index()
)
pivot.columns.name = None
pivot = pivot.rename(columns={1: 'male', 2: 'female'})
pivot['total'] = pivot['male'] + pivot['female']

# Identify unisex names (5–95% for either sex, total > 1,000)
mixed_stats = pivot.groupby('preusuel').agg(
    masc=('male', 'sum'),
    fem=('female', 'sum'),
).reset_index()
mixed_stats['total'] = mixed_stats['masc'] + mixed_stats['fem']
mixed_stats['prop_f'] = mixed_stats['fem'] / mixed_stats['total']

mixed_names = (
    mixed_stats[
        (mixed_stats['prop_f'] > 0.05) &
        (mixed_stats['prop_f'] < 0.95) &
        (mixed_stats['total'] > 1000)
    ]['preusuel']
    .sort_values()
    .tolist()
)

# Long format for stacked area
viz3_base = pivot[pivot['preusuel'].isin(mixed_names)].copy()
viz3_long = viz3_base.melt(
    id_vars=['preusuel', 'annais', 'total'],
    value_vars=['male', 'female'],
    var_name='gender',
    value_name='count',
)
viz3_long['proportion'] = (viz3_long['count'] / viz3_long['total']).round(4)

print(f"Unisex names identified: {len(mixed_names)}")

Unisex names identified: 55


In [8]:
# Dropdown — unisex names only
dropdown3 = alt.binding_select(options=mixed_names, name='Name: ')
name_select3 = alt.selection_point(
    fields=['preusuel'],
    bind=dropdown3,
    value=mixed_names[0],
)

color_scale = alt.Scale(
    domain=['male', 'female'],
    range=['#4C72B0', '#DD8452'],
)

# Stacked area 100%
area = (
    alt.Chart(viz3_long)
    .mark_area()
    .encode(
        x=alt.X(
            'annais:O',
            title='Year',
            axis=alt.Axis(values=list(range(1900, 2021, 10)), labelAngle=-45),
        ),
        y=alt.Y(
            'proportion:Q',
            stack='normalize',
            title='Proportion',
            axis=alt.Axis(format='%'),
        ),
        color=alt.Color('gender:N', scale=color_scale, title='Gender'),
        order=alt.Order('gender:N', sort='descending'),
        tooltip=[
            alt.Tooltip('preusuel:N',   title='Name'),
            alt.Tooltip('annais:O',     title='Year'),
            alt.Tooltip('gender:N',     title='Gender'),
            alt.Tooltip('count:Q',      title='Births', format=','),
            alt.Tooltip('proportion:Q', title='Proportion', format='.1%'),
        ],
    )
    .add_params(name_select3)
    .transform_filter(name_select3)
    .properties(width=860, height=380)
)

# Reference line at 50%
import pandas as pd
rule = (
    alt.Chart(pd.DataFrame({'y': [0.5]}))
    .mark_rule(color='white', strokeDash=[6, 3], strokeWidth=1.5)
    .encode(y='y:Q')
)

chart_viz3 = (area + rule).properties(
    title=alt.TitleParams(
        'Male / female breakdown of a name over time (1900–2020)',
        fontSize=14,
    )
)

chart_viz3

alt.LayerChart(...)